In [1]:
from Bio import SeqIO

def filter_fasta_by_length(inp, out, min_len=0, max_len=None, molecule_type="auto"):
    valid = {"protein": set("ACDEFGHIKLMNPQRSTVWYBXZJUO*"),
             "dna": set("ACGTUNRYKMSWBDHV-"),
             "rna": set("ACGUNRYKMSWBDHV-")}
    kept = []
    for rec in SeqIO.parse(inp, "fasta"):
        L = len(rec.seq)
        if L < min_len or (max_len is not None and L > max_len):
            continue
        if molecule_type != "auto" and (set(str(rec.seq).upper()) - valid[molecule_type]):
            continue
        kept.append(rec)
    SeqIO.write(kept, out, "fasta")
    print(f"{out}: {len(kept)} seqs")

In [2]:
filter_fasta_by_length('orfs_1B_removed_bastro.faa', 'orfs_1B_removed_bastro_min_333_len.faa', 333)

orfs_1B_removed_bastro_min_333_len.faa: 67 seqs


In [4]:
import re
import pandas as pd
from Bio import SeqIO

df = pd.read_csv(
    "paired_refseq_viral_and_astro_orfs_1_50_batches_results.tsv",
    sep="\t",
    dtype=str,
)

def parse_info(info):
    m = re.search(r"Contig_ID:\s*([^|]+)", info or "")
    return m.group(1).strip() if m else None

df["_contig"] = df["Инфо"].map(parse_info)
lookup = {(r["Образец"], r["_contig"]): r for _, r in df.iterrows()}

records = []
for rec in SeqIO.parse("orfs_1B_removed_bastro_min_333_len.faa", "fasta"):
    m = re.match(r"^([^|]+)\|([^/]+)", rec.description)
    if not m:
        records.append(rec)
        continue
    sample, contig = m.group(1), m.group(2)
    row = lookup.get((sample, contig))
    if row is None:
        records.append(rec)
        continue
    new = f"{sample}/{contig}/{row['Страна']}/{row['Год']}/{row['Хозяин']}"
    rec.description = new
    rec.id = new.split()[0]
    records.append(rec)

SeqIO.write(records, "orfs_1B_removed_bastro_min_333_len_headers.faa", "fasta")
print(f"Записано {len(records)} последовательностей")

Записано 67 последовательностей
